### Create basic tsv of lang meta info

In [12]:
from iso639 import Lang

In [13]:
import os

fleurs_dir = "fleurs/data/"
lang_dirs = os.listdir(fleurs_dir)

In [16]:
lang_meta = {'dir': [], 'name':[], 'iso639-1': [], 'iso639-3': [], 'script': []}
for dr in lang_dirs:
    pth = os.path.join(fleurs_dir, dr)
    if os.path.isdir(pth):
        name = dr.split('_')
        lang = Lang(name[0])
        lang_meta['dir'].append(pth)
        lang_meta['name'].append(lang.name)
        lang_meta['iso639-1'].append(lang.pt1)
        lang_meta['iso639-3'].append(lang.pt3)
        if len(name) == 3:
            lang_meta['script'].append(name[1])
        else:
            lang_meta['script'].append('')
        

In [17]:
import pandas as pd

lang_meta_df = pd.DataFrame(lang_meta)
lang_meta_df

,dir,name,iso639-1,iso639-3,script
0,fleurs/data/af_za,Afrikaans,af,afr,
1,fleurs/data/am_et,Amharic,am,amh,
2,fleurs/data/ar_eg,Arabic,ar,ara,
3,fleurs/data/as_in,Assamese,as,asm,
4,fleurs/data/ast_es,Asturian,,ast,
...,...,...,...,...,...
97,fleurs/data/wo_sn,Wolof,wo,wol,
98,fleurs/data/xh_za,Xhosa,xh,xho,
99,fleurs/data/yo_ng,Yoruba,yo,yor,
100,fleurs/data/yue_hant_hk,Yue Chinese,,yue,hant


In [19]:
lang_meta_df = pd.read_csv("lang_meta.tsv", sep='\t').fillna('')

In [20]:
lang_meta_df

,dir,name,iso639-1,iso639-3,script,northeuralex,dict,g2p,phoneset,acoustic
0,fleurs/data/af_za,Afrikaans,af,afr,,,,,,
1,fleurs/data/am_et,Amharic,am,amh,,,,,,
2,fleurs/data/ar_eg,Arabic,ar,ara,,,arabic_mfa,,,
3,fleurs/data/as_in,Assamese,as,asm,,,,,,
4,fleurs/data/ast_es,Asturian,,ast,,,,,,
...,...,...,...,...,...,...,...,...,...,...
97,fleurs/data/wo_sn,Wolof,wo,wol,,,,,,
98,fleurs/data/xh_za,Xhosa,xh,xho,,,,,,
99,fleurs/data/yo_ng,Yoruba,yo,yor,,,,,,
100,fleurs/data/yue_hant_hk,Yue Chinese,,yue,hant,,,,,


In [21]:
north_eura_lex_langs = pd.read_csv("northeuralex/northeuralex-0.9-language-data.tsv",sep='\t').iso_code.tolist()

In [25]:
lang_meta_df['northeuralex'] = lang_meta_df.apply(lambda x: x['iso639-3'] if x['iso639-3'] in north_eura_lex_langs else '', axis=1)

In [27]:
#lang_meta_df.to_csv("lang_meta.tsv", sep='\t', index=False)

### Create text files from train tsv

In [2]:
hundreds_map = {
    'af_za': 'honderd',
    'as_in': 'শ',            # Assamese (as in উনিশ শ ...)
    'bn_in': 'শো',           # Bengali (উনিশশো ...)
    'da_dk': 'hundrede',
    'de_de': 'hundert',      # German (neunzehnhundert ...)
    'gu_in': 'સો',           # Gujarati (ઓગણીસ સો ...)
    'hi_in': 'सौ',           # Hindi (उन्नीस सौ ...)
    'is_is': 'hundruð',      # Icelandic (nítján hundruð ...)
    'kn_in': 'ನೂರು',         # Kannada (used in compound hundreds)
    'lb_lu': 'honnert',      # Luxembourgish
    'ml_in': 'നൂറ്',         # Malayalam (bound form appears in compounds)
    'mr_in': 'शे',           # Marathi (एकोणीसशे ...)
    'ne_np': 'सय',           # Nepali (उन्नाइस सय ...)
    'nl_nl': 'honderd',      # Dutch (negentienhonderd ...)
    'or_in': 'ଶହ',           # Odia (ଊଣେଇଶ ଶହ ...)
    'pa_in': 'ਸੌ',           # Punjabi (ਉੱਨੀ ਸੌ ...)
    'sd_in': 'سو',           # Sindhi
    'sv_se': 'hundra',       # Swedish (nittonhundra ...)
    'ta_in': 'நூற்று',        # Tamil (bound form)
    'te_in': 'వందల',         # Telugu (bound plural form)
    'ur_pk': 'سو',            # Urdu (انیس سو ...)
    'en_us': 'hundred',
    'nb_no': 'hundre',
}

In [18]:
from tqdm import tqdm
import pandas as pd
import os
import csv
import unicodedata
from num2words import num2words
import re

root = "fleurs/data"
out_root = "fleurs_ipa/"
HEADERS = ['id', 'audio_file', 'text', 'text_normalized', 'chars', 'speaker_id', 'gender']
LATIN_LANG_PREFIXES = (
    "af_", "ast_", "az_", "bs_", "ca_", "ceb_", "cs_", "cy_", "da_", "de_", "en_", "es_", "et_", "ff_", "fi_", "fil_", "fr_", "ga_", "gl_", "ha_", "hr_", "hu_", "id_", "ig_", "is_", "it_", "jv_", "kam_", "kea_", "lb_", "lg_", "ln_", "lt_", "luo_", "lv_", "mi_", "ms_", "mt_", "nb_", "nl_", "nso_", "ny_", "oc_", "om_", "pl_", "pt_", "ro_", "sk_", "sl_", "sn_", "so_", "sv_", "sw_", "tr_", "umb_", "uz_", "vi_", "wo_", "xh_", "yo_", "zu_"
)

ZH_PREFIXES = ("cmn_","yue_")


def decide_mode(n):
    if 1200 <= n <= 2050:
        return "year"
    return "cardinal"

def apply_num2words(n, lang):
    if not lang:
        return str(n)
    try:
        text = num2words(n, lang=lang)
    except:
        print(n, lang)
    text = re.sub(r'[.,]', '', text)
    text = re.sub(r'[-]', ' ', text)
    return text
    
flip_mode = {'year': 'cardinal', 'cardinal': 'year'}
def repl(m, n2w_code, lang_code, year_type):
    num_str = m.group()
    n = int(num_str)
    mode = decide_mode(n)
    n2w = lambda x: apply_num2words(x, lang=n2w_code)
    if mode == 'year':
        if year_type == 3:
            return n2w(n)
    
        elif year_type == 4:
            return ' '.join([n2w(int(d)) for d in num_str])
    
        else:
            if n >= 2000:
                return n2w(n)
    
            first = int(num_str[:2])
            last_num = int(num_str[2:])
    
            if year_type == 2:  # English type
                if last_num == 0:
                    return f"{n2w(first)} {hundreds_map[lang_code]}"
                elif last_num < 10:
                    return f"{n2w(first)} {hundreds_map[lang_code]} {n2w(last_num)}"
                else:
                    return f"{n2w(first)} {n2w(last_num)}"
    
            else:  # Type (1)
                parts = [n2w(first), hundreds_map[lang_code]]
                if last_num != 0:
                    parts.append(n2w(last_num))
                return ' '.join(parts)

    return n2w(n)
    

def clean_text(text, lang_dir, n2w_lang='', year_type=3):
    # remove punctuation
    text = "".join(ch for ch in text if not unicodedata.category(ch).startswith("P"))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Cf')

    # collapse spaces BETWEEN CJK characters
    is_zh = lang_dir.startswith(ZH_PREFIXES)
    if is_zh:
        text = re.sub(
            r'(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])',
            '',
            text
        )
        text = re.sub(r'[—–―─]+', ' ', text)
    is_latin_lang = lang_dir.startswith(LATIN_LANG_PREFIXES)
    # ONLY apply this for non-Latin languages
    if not is_latin_lang:
        text = re.sub(
            r'(?<=[A-Za-z\u00C0-\u024F])(?=[^A-Za-z\u00C0-\u024F\s])|(?<=[^A-Za-z\u00C0-\u024F\s])(?=[A-Za-z\u00C0-\u024F])',
            ' ',
            text
        )

    # separate digits and non-digits (safe for all)
    text = re.sub(r'(?<=\d)(?=\D)|(?<=\D)(?=\d)', ' ', text)

    # Currency normalization
    text = re.sub(r'us\s*\$\s*(147)\s+(\S+)', r'\1 \2 us dollars ', text)
    
    text = re.sub(
            r'\b(us|aud|vs)\s*\$\s*(\d+(?:\s+\d+)*)',
            lambda m: f" {re.sub(r'\s+', '', m.group(2))} {m.group(1).lower()} dollars ",
            text,
            flags=re.IGNORECASE
        )
    text = re.sub(
            r'\$\s*(\d+(?:\s+\d+)*)',
            lambda m: f" {re.sub(r'\s+', '', m.group(1))} dollars ",
            text
        )
    text = re.sub(r'¥\s*([\d\s]+)', r'\1 yen ', text)
    text = re.sub(r'£\s*([\d\s]+)', r'\1 pounds ', text)

    # misc signs
    text = re.sub(r'\butc\s*\+\s*(\d+)', r'utc plus \1', text, flags=re.IGNORECASE)
    text = re.sub(r' \+ ', r' plus ', text, flags=re.IGNORECASE)
    text = re.sub(r'°\s*([cfw])\b', 
              lambda m: {'c': 'degrees celsius ',
                         'f': 'degrees fahrenheit ',
                         'w': 'degrees watts '}[m.group(1).lower()],
              text, flags=re.IGNORECASE)

    # time 0919 -> 9 19
    text = re.sub(r'\b0(\d{3})\b', lambda m: f"{int(m.group(1)[:1])} {int(m.group(1)[1:])}", text)
    # time 1119 -> 11 19
    text = re.sub(r'\b1([0-1])(\d{2})\b', lambda m: f"1{int(m.group(1))} {int(m.group(2))}" if int(m.group(2)) != 0 else f"1{m.group(1)}{m.group(2)}", text)
    # two years
    text = re.sub(r'\b(\d{8})\b', lambda m: f"{m.group(1)[:4]} {m.group(1)[4:]}" if all(decide_mode(int(x)) == "year" for x in (m.group(1)[:4], m.group(1)[4:])) else m.group(1), text)

    # expand numbers
    text = re.sub(r'\d+', lambda x: repl(x, n2w_lang, lang_dir, year_type), text)
    # normalize whitespace
    text = " ".join(text.split())

    return text

meta_df = pd.read_csv("lang_meta.tsv", sep='\t').fillna('')
for lang_dir in tqdm(os.listdir(root)):
    lang_path = os.path.join(root, lang_dir)
    if not os.path.isdir(lang_path):
        continue
        
    n2w_id = meta_df[meta_df['dir']==lang_path]['num2words'].tolist()[0]
    year_type = meta_df[meta_df['dir']==lang_path]['year'].tolist()[0]
    out_path = os.path.join(out_root, lang_dir)
    os.makedirs(out_path, exist_ok=True)
    tsv_path = os.path.join(lang_path, "train.tsv")
    try:
        lang_df = pd.read_csv(tsv_path, names=HEADERS, sep='\t', quoting=csv.QUOTE_NONE, encoding='utf-8')
    except:
        lang_df = pd.read_csv(tsv_path, names=HEADERS+[""], sep='\t', quoting=csv.QUOTE_NONE, encoding='utf-8')
    lang_text = lang_df['text_normalized'].tolist()
    lang_text = [clean_text(t, lang_dir, n2w_lang=n2w_id, year_type=year_type) for t in lang_text]
    with open(os.path.join(out_path, 'train_sentences.txt'), 'w') as fp:
        fp.write('\n'.join(lang_text))

100%|█████████████████████████████████████████| 103/103 [00:21<00:00,  4.80it/s]


### Create Epitran transcriptions

In [186]:
import epitran
import pandas as pd
import os
from tqdm import tqdm
import logging
import re

# Mandarin
import jieba

# Cantonese
import pycantonese 

# Japanese
from fugashi import Tagger
ja_tagger = Tagger()  # fugashi / MeCab
import pyopenjtalk

# Thai
from pythainlp.tokenize import word_tokenize as thai_tokenize

# Lao
from laonlp.tokenize import word_tokenize as lao_tokenize

# Vietnamese
from pyvi import ViTokenizer


# Khmer tokenizer
from khmerns import tokenize as km_tokenize, normalize as km_normalize

# Burmese
from pyidaungsu import tokenize as mm_tokenize

# Hebrew
from phonikud_onnx import Phonikud
from phonikud import phonemize

he_model = Phonikud("epitran_dicts/phonikud-1.0.int8.onnx")

he_phonemize = lambda text: phonemize(he_model.add_diacritics(text))


logger = logging.Logger('catch_all')

tqdm.pandas()

In [187]:
def merge_ascii_sequences(tokens):
    merged = []
    buffer = []

    for tok in tokens:
        # ASCII letter or digit
        if re.match(r'^[A-Za-z0-9]$', tok):
            buffer.append(tok)
        else:
            if buffer:
                merged.append(''.join(buffer))
                buffer = []
            merged.append(tok)

    # flush buffer
    if buffer:
        merged.append(''.join(buffer))

    return merged

#### Japanese G2P

In [188]:
def normalize_tokens(tokens):
    fixed = []

    for t in tokens:
        # split pau merged tokens
        if t.startswith('pau') and t != 'pau':
            fixed.append('pau')
            rest = t[3:]
            if rest:
                fixed.append(rest)
            continue

        fixed.append(t)

    return fixed

PALATAL_MAP = {
    'k': 'kʲ',
    'g': 'ɡʲ',
    'n': 'nʲ',
    'h': 'ç',
    'b': 'bʲ',
    'p': 'pʲ',
    'm': 'mʲ',
    'r': 'ɾʲ',
    't': 'tʲ',
    'd': 'dʲ',
}

# -------------------------
# Core phoneme mapping
# -------------------------
PHONEME_MAP = {
    # vowels
    'a': 'a',
    'i': 'i',
    'u': 'ɯ',
    'U': 'ɯ̥',   # devoiced u
    'e': 'e',
    'o': 'o',

    # consonants
    'k': 'k',
    'g': 'ɡ',
    's': 's',
    'z': 'z',
    't': 't',
    'd': 'd',
    'n': 'n',
    'h': 'h',
    'b': 'b',
    'p': 'p',
    'm': 'm',
    'y': 'j',
    'r': 'ɾ',
    'w': 'w',

    # special consonants
    'sh': 'ɕ',
    'ch': 'tɕ',
    'ts': 'ts',
    'j': 'dʑ',
    'f': 'ɸ',

    # nasal mora
    'N': 'ɴ',

    # gemination (促音)
    'cl': 'ʔ',
    
    # devoiced vowel
    'I': 'i̥',

    # rare /v/ (loanwords)
    'v': 'v',

}

# -------------------------
# Combine tokens like "k y a" → "kya"
# -------------------------
def combine_glides(tokens):
    combined = []
    i = 0

    while i < len(tokens):
        # Case 1: C + y + V → palatalized syllable
        if (
            i < len(tokens) - 2 and
            tokens[i+1] == 'y' and
            tokens[i+2] in ['a', 'u', 'o']
        ):
            combined.append(tokens[i] + 'y' + tokens[i+2])
            i += 3
            continue

        # Case 2: V + y + V → glide vowel (iya, ayo, etc.)
        if (
            i < len(tokens) - 2 and
            tokens[i] in ['a','i','u','e','o'] and
            tokens[i+1] == 'y' and
            tokens[i+2] in ['a','u','o']
        ):
            combined.append(tokens[i] + 'y' + tokens[i+2])
            i += 3
            continue

        combined.append(tokens[i])
        i += 1

    return combined



def map_token(t):
    # palatalized consonants: ky, gy, etc.
    if len(t) == 2 and t[1] == 'y' and t[0] in PALATAL_MAP:
        return PALATAL_MAP[t[0]]

    # ky + vowel (kya, kyu, kyo)
    if len(t) == 3 and t[1] == 'y':
        base = t[0]
        vowel = t[2]

        if base in PALATAL_MAP and vowel in PHONEME_MAP:
            return PALATAL_MAP[base] + PHONEME_MAP[vowel]

    # iya, ayo, etc.
    if len(t) == 3 and t[1] == 'y':
        if t[0] in PHONEME_MAP and t[2] in PHONEME_MAP:
            return PHONEME_MAP[t[0]] + 'j' + PHONEME_MAP[t[2]]

    return PHONEME_MAP.get(t, t)
    
# -------------------------
# Convert OpenJTalk → IPA
# -------------------------
def openjtalk_to_ipa(text):

    tokens = text.strip().split()

    # Step 1: combine glide sequences
    tokens = combine_glides(tokens)

    ipa = []
    i = 0

    while i < len(tokens):
        t = tokens[i]

        # pause → space
        if t in ['|']:
            ipa.append(' ')
            i += 1
            continue

        # gemination handling (cl)
        if t == 'cl':
            # lookahead → double consonant
            if i + 1 < len(tokens):
                nxt = tokens[i + 1]
                if nxt in PHONEME_MAP:
                    cons = PHONEME_MAP[nxt]
                    ipa.append(cons)  # geminated consonant
                else:
                    ipa.append('ʔ')
            else:
                ipa.append('ʔ')
            i += 1
            continue

        # long vowel handling
        if (
            i < len(tokens) - 1 and
            tokens[i] in ['a', 'i', 'u', 'U', 'e', 'o', 'I'] and
            tokens[i] == tokens[i + 1]
        ):
            ipa.append(PHONEME_MAP[t] + 'ː')
            i += 2
            continue

        # normal mapping
        ipa.append(map_token(t))

        i += 1

    # clean spaces
    out = ''.join(ipa)
    out = re.sub(r'\s+', ' ', out).strip()

    return out

#### Khmer g2p

In [189]:
from lingpy import ipa2tokens, tokens2class

DIACRITIC_MAP = {
    'ា': 'aː',
    'ិ': 'i',
    'ី': 'iː',
    'ឹ': 'ɨ',
    'ឺ': 'ɨː',
    'ុ': 'u',
    'ូ': 'uː',
    'ួ': 'uə',
    'េ': 'eː',
    'ែ': 'ɛː',
    'ៃ': 'ɨj',
    'ោ': 'oː',
    'ៅ': 'ɨw',
    'ំ': 'am',
    'ះ': 'ah',
    'ើ': 'əː',
    'ឿ': 'ɨə',
    'ៀ': 'iə',
    '់': '',   # often coda/shortening marker
}

KHMER_INDEPENDENT_VOWELS = {
    'ឥ': 'ʔe',
    'ឦ': 'ʔei',
    'ឧ': 'ʔo',
    'ឨ': 'ʔa',
    'ឩ': 'ʔuː',
    'ឪ': 'ʔaw',
    'ឫ': 'rɨ',
    'ឬ': 'rɨː',
    'ឭ': 'lɨ',
    'ឮ': 'lɨː',
    'ឯ': 'ae',
    'ឰ': 'aj',
    'ឱ': 'ao',
    'ឳ': 'aw',
}

def fix_khmer_ipa(ipa):
    ipa = ''.join(KHMER_INDEPENDENT_VOWELS.get(ch, ch) for ch in ipa)
    if ipa == '':
        return ''
    tokens = ipa2tokens(ipa, merge_vowels=False)
    try:
        classes = tokens2class(tokens, 'dolgo')
    except:
        return ipa

    fixed = []
    vowel_count = 0
    i = 0

    while i < len(tokens):
        tok = tokens[i]
        cls = classes[i]
        if cls == 'V':
            vowel_count += 1

        # if diacritic (class '0')
        if cls == '0':
            if fixed:
                prev = fixed.pop()
                # replace previous vowel if possible
                replacement = DIACRITIC_MAP.get(tok, '')

                if replacement:
                    fixed.append(replacement)
                else:
                    fixed.append(prev)  # fallback
            i += 1
            continue

        fixed.append(tok)
        i += 1
    if fixed[-1] in ['ɑː', 'ɔː'] and vowel_count >= 2:
        fixed = fixed[:-1]
    return ''.join(fixed)

#### Burmese g2p

In [190]:
BURMESE_DIACRITIC_MAP = {
    'ါ': 'aː',
    'ာ': 'aː',

    'ှ': 'ʰ',      # medial aspiration (attach to consonant)

    '္': '',       # kill inherent vowel (handled structurally)

    '့': 'ˀ',      # creaky tone (simple marker)
    'း': 'ː',      # length / high tone (approx)

    'ဿ': 's',      # safe simplification

    'ၤ': 'ŋ',      # nasal coda
}

def fix_burmese_ipa(ipa):
    if ipa == '':
        return ''

    tokens = ipa2tokens(ipa, merge_vowels=False)

    try:
        classes = tokens2class(tokens, 'dolgo')
    except:
        return ipa

    fixed = []
    i = 0

    while i < len(tokens):
        tok = tokens[i]
        cls = classes[i]

        # 🔥 handle Burmese diacritics explicitly
        if cls == '0':

            replacement = BURMESE_DIACRITIC_MAP.get(tok, '')

            if not fixed:
                i += 1
                continue

            # --- CASE 1: vowel replacement ---
            if tok in ['ါ', 'ာ']:
                # replace nearest previous vowel
                for j in range(len(fixed) - 1, -1, -1):
                    if tokens2class([fixed[j]], 'dolgo')[0] == 'V':
                        fixed[j] = replacement
                        break

            # --- CASE 2: medial (attach to consonant) ---
            elif tok == 'ှ':
                fixed[-1] += replacement

            # --- CASE 3: virama (remove inherent vowel) ---
            elif tok == '္':
                if fixed and tokens2class([fixed[-1]], 'dolgo')[0] == 'V':
                    fixed.pop()

            # --- CASE 4: tone marks ---
            elif tok in ['့', 'း']:
                if replacement not in fixed[-1]:
                    fixed[-1] += replacement

            # --- CASE 5: nasal ---
            elif tok == 'ၤ':
                fixed[-1] += replacement

            # --- fallback ---
            else:
                fixed[-1] += replacement

            i += 1
            continue

        # normal token
        fixed.append(tok)
        i += 1

    return ''.join(fixed)

#### Persian g2p

In [191]:
PERSIAN_MAP = {
    # letters
    'پ': 'p',
    'ک': 'k',
    'ی': 'i',
    'ي': 'i',
    'ہ': 'h',
    'ھ': 'ʰ',
    'ە': 'e',

    # vowels
    'َ': 'a',
    'ِ': 'e',
    'ُ': 'o',

    # tanween / nasal
    'ً': 'an',
    'ٍ': 'en',

    # nasalization mark
    '◌̃': '̃',

    # special forms
    'ۃ': 'a',
    'ۂ': 'e',

    # ignore
    'ٓ': '',
    'ٔ': '',
    'ٕ': '',
    'ٴ': '',
}

def fix_persian_ipa(ipa):
    if ipa == '':
        return ''

    tokens = ipa2tokens(ipa, merge_vowels=False)

    try:
        classes = tokens2class(tokens, 'dolgo')
    except:
        return ipa

    fixed = []
    i = 0

    while i < len(tokens):
        tok = tokens[i]
        cls = classes[i]

        # 🔥 handle non-IPA / diacritics
        if cls == '0' or tok in PERSIAN_MAP:
            replacement = PERSIAN_MAP.get(tok, None)

            # skip explicitly ignored
            if replacement == '':
                i += 1
                continue

            # --- CASE 1: diacritic attaches to previous ---
            if tok in ['َ', 'ِ', 'ُ', 'ّ', 'ً', 'ٍ', '◌̃']:
                if fixed:
                    if tok == 'ّ':  # gemination
                        fixed.append(fixed[-1])
                    elif tok == '◌̃':
                        fixed[-1] += '̃'
                    else:
                        fixed.append(replacement)
                else:
                    fixed.append(replacement)

            # --- CASE 2: normal letter replacement ---
            elif replacement is not None:
                fixed.append(replacement)

            i += 1
            continue

        # normal IPA token
        fixed.append(tok)
        i += 1

    return ''.join(fixed)

In [232]:
meta_pd = pd.read_csv('lang_meta.tsv', sep='\t').fillna('')
loaded = False

def transcribe_epi(meta_row):
    lang = os.path.basename(meta_row['dir'])
    lang_pth = os.path.join('fleurs_ipa', lang)
    epi_code = meta_row['epitran']

    if not epi_code in ['yue-Hant']:
        return
    if epi_code == 'yue-Hant':
        global loaded
        if not loaded:
            jieba.load_userdict("epitran_dicts/cccanto-170202/cccanto-webdist.txt")
            loaded = True
            
    if epi_code:
        try:
            if 'cmn' in epi_code:
                epi = epitran.Epitran(epi_code, cedict_file='epitran_dicts/cedict_1_0_ts_utf-8_mdbg/cedict_ts.u8', tones=True)
            elif 'yue' in epi_code:
                epi = epitran.Epitran(epi_code, cedict_file='epitran_dicts/cccanto-170202/cccanto-webdist.txt', tones=True)
            elif 'heb' in epi_code:
                epi = None
            else:
                epi = epitran.Epitran(epi_code, tones=True)
        except Exception as e:
            print(epi_code)
            logger.exception(e)
            return

        with open(os.path.join(lang_pth, 'train_sentences.txt'), 'r', encoding='utf-8') as fp:
            lang_text = fp.read().split('\n')

        lang_text_transcribed = []
        lang_text_reversed = []  # NEW: for MFA input

        for line in lang_text:
           
            if any(x in epi_code for x in ['jpn', 'tha', 'lao', 'khm', 'mya', 'cmn', 'yue', 'vie']):

                # 🔥 Segment FIRST
                if 'cmn' in epi_code:
                    tokens = [t for t in jieba.cut(line) if t]
                elif 'yue' in epi_code:
                    #tokens = pycantonese.segment(line)
                    tokens = [t for t in jieba.cut(line) if t]
                elif 'jpn' in epi_code:
                    tokens = [word.surface for word in ja_tagger(line)]
                elif 'tha' in epi_code:
                    tokens = thai_tokenize(line)
                elif 'lao' in epi_code:
                    tokens = lao_tokenize(line)
                elif 'vie' in epi_code:
                    tokens = ViTokenizer.tokenize(line).split()
                elif 'khm' in epi_code:
                    tokens = km_tokenize(km_normalize(line))
                    tokens = merge_ascii_sequences([t.strip() for t in tokens])
                elif 'mya' in epi_code:
                    tokens = mm_tokenize(line, form='word')
                    text = ' '.join(tokens)
                    text = re.sub(
                            r'(?<=[A-Za-z\u00C0-\u024F])(?=[^A-Za-z\u00C0-\u024F\s])|(?<=[^A-Za-z\u00C0-\u024F\s])(?=[A-Za-z\u00C0-\u024F])',
                            ' ',
                            text
                        )
                
                    # separate digits and non-digits (safe for all)
                    text = re.sub(r'(?<=\d)(?=\D)|(?<=\D)(?=\d)', ' ', text)
                
                    # normalize whitespace
                    tokens = text.split()
                else:
                    tokens = line.split()
                    
                segmented = ' '.join([t.strip() for t in tokens])  # fallback only


                if 'jpn' in epi_code:
                    ipa = ' '.join([openjtalk_to_ipa(pyopenjtalk.g2p(w)) for w in segmented.split()])
                    lang_text_transcribed.append(ipa)
                elif 'khm' in epi_code:
                    segmented = re.sub(r'(\S+)\s*ៗ', r'\1 \1', segmented)
                    segmented = segmented.replace('៎','')
                    ipa = epi.transliterate(segmented)
                    ipa = [fix_khmer_ipa(w) for w in ipa.split(' ')]
                    lang_text_transcribed.append(' '.join(ipa))
                elif 'mya' in epi_code:
                    ipa = epi.transliterate(segmented)
                    ipa = [fix_burmese_ipa(w) for w in ipa.split(' ')]
                    lang_text_transcribed.append(' '.join(ipa))
                elif 'lao' in epi_code:
                    segmented = re.sub(r'(\S+)\s*ໆ', r'\1 \1', segmented)
                    ipa = epi.transliterate(segmented)
                    ipa = ipa.replace('ຼ', '').replace('໌', '')
                    lang_text_transcribed.append(ipa)
                elif 'tha' in epi_code:
                    segmented = re.sub(r'(\S+)\s*ๆ', r'\1 \1', segmented)
                    ipa = epi.transliterate(segmented)
                    lang_text_transcribed.append(ipa)
                elif 'vie' in epi_code:
                    # Step 1: expand to syllables
                    flat_tokens = segmented.replace('_', ' ').split()
                    # Step 2: G2P per syllable
                    ipa_tokens = [epi.transliterate(t) for t in flat_tokens]
                    # Step 3: reconstruct words
                    words = segmented.split()
                    result = []
                    i = 0
                    for word in words:
                        n = word.count('_') + 1
                        chunk = ipa_tokens[i:i+n]
                        result.append('_'.join(chunk))
                        i += n
                    ipa = ' '.join(result)
                    # optional cleanup
                    ipa = ipa.replace('_', '')
                    lang_text_transcribed.append(ipa)
                else:
                    ipa = epi.transliterate(segmented)
                    lang_text_transcribed.append(ipa)
                lang_text_reversed.append(segmented)
            elif 'heb' in epi_code:
                ipa = he_phonemize(line)
                
                lang_text_transcribed.append(ipa)
                lang_text_reversed.append(line)
            else:
                ipa = epi.transliterate(line)
                if 'mal' in epi_code:
                    ipa = ipa.replace("a്","").replace("്","")
                elif any(x in epi_code for x in ['fas', 'urd', 'pbu', 'ara']):
                    ipa = ' '.join([fix_persian_ipa(w) for w in ipa.split(' ')])
                elif 'dan' in epi_code:
                    ipa = ipa.replace('̊', '')

                lang_text_transcribed.append(ipa)
                lang_text_reversed.append(line)

        # write transcription
        with open(os.path.join(lang_pth, 'train_sentences_epitran.txt'), 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(lang_text_transcribed))

        # NEW: write reversed input for MFA
        with open(os.path.join(lang_pth, 'train_sentences_input.txt'), 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(lang_text_reversed))
    

In [233]:
meta_pd.progress_apply(transcribe_epi, axis=1)

100%|█████████████████████████████████████████| 102/102 [00:06<00:00, 16.27it/s]


0      None
1      None
2      None
3      None
4      None
       ... 
97     None
98     None
99     None
100    None
101    None
Length: 102, dtype: object

### Create pronunciation dictionaries

In [245]:
from lingpy import ipa2tokens, tokens2class
import re
import epitran
import unicodedata


epi_en = epitran.Epitran('eng-Latn')
LATIN_RE = re.compile(r'[a-z]+')

dyn = {}

def shift_chunk(tokens, classes):
    key = tuple(tokens)  # faster + hashable
    if key in dyn:
        return dyn[key]

    tokens = list(tokens)  # local copy

    for i, cls in enumerate(classes):
        if cls in ['0', '1']:

            # already after vowel → OK
            if i > 0 and classes[i - 1] in ['V', 'R']:
                continue

            # find nearest previous vowel
            for j in range(i - 1, -1, -1):
                if classes[j] in ['V', 'R']:
                    tone = tokens.pop(i)
                    tokens.insert(j + 1, tone)
                    break
            break  # only one tone per chunk

    dyn[key] = tokens
    return tokens

def split_by_tone(tokens, classes):
    chunks = []
    current = []

    for t, c in zip(tokens, classes):
        current.append(t)
        if c in ['0', '1']:  # tone marker ends syllable
            chunks.append(current)
            current = []

    if current:
        chunks.append(current)

    return chunks
    
def shift_tone_to_vowel(tokens):
    try:
        classes = tokens2class(tokens, 'dolgo')
    except:
        #print(tokens)
        return []

    chunks = split_by_tone(tokens, classes)

    result = []

    for chunk in chunks:
        try:
            c_classes = tokens2class(chunk, 'dolgo')
        except:
            result.extend([])
            continue
        shifted = shift_chunk(chunk, c_classes)
        result.extend(shifted)

    return result

def build_lexicon_entry(src_sent: str, tar_sent: str, toned=False, latn=True):
    """
    Build a partial pronunciation dictionary from a source sentence
    and its G2P-transcribed target sentence.

    Returns:
        dict: {word: set(pronunciations)}
    """

    src_tokens = re.sub(r"\s", " ", src_sent).split(' ')
    tar_tokens = re.sub(r"\s", " ", tar_sent).split(' ')

    lexicon = {}

    if len(src_tokens) != len(tar_tokens):
        print(src_tokens)
        print(tar_tokens)
        raise ValueError("Source and target token lengths do not match")

    for src_tok, tar_tok in zip(src_tokens, tar_tokens):

        # Case 1: empty pronunciation (e.g., silent word)
        

        if src_tok.strip() == "":
            continue
        
        tar_tok = tar_tok.replace("@","")
        # Case 2: entirely digits → treat as unknown
        if src_tok.isdigit():
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 3: pure latin lowercase ASCII and Latin based orthography
        if LATIN_RE.fullmatch(tar_tok) and latn:
            tokens = ipa2tokens(tar_tok, merge_vowels=False)
            lexicon.setdefault(src_tok, set()).add(' '.join(tokens))
            continue
            
        # Case 3.5: pure latin lowercase ASCII and not Latin based orthograpy -> change to English pronunciations
        if LATIN_RE.fullmatch(src_tok) and not latn:
            tokens = ipa2tokens(epi_en.transliterate(src_tok), merge_vowels=False)
            lexicon.setdefault(src_tok, set()).add(' '.join(tokens))
            continue
        
        # Case 4: transliteration failed (non-latin unchanged or leaked)
        if (src_tok == tar_tok) and not latn:
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 5: obvious failure marker (e.g., @@@@)
        if tar_tok == "":
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        # Case 6: non-latin scripts
        tokens = ipa2tokens(tar_tok, merge_vowels=False)

            
        if toned:
            tokens = shift_tone_to_vowel(tokens)

        if not tokens:
            lexicon.setdefault(src_tok, set()).add("spn")
            continue

        lexicon.setdefault(src_tok, set()).add(' '.join(tokens))


    return lexicon

In [246]:
src1 = "35 mile ఆకృతి వాస్తవానికి కొంత గందరగోళంగా 36 mm వెడల్పు 24 mm ఎత్తు"
tar1 = "@@ m@@@ aːkrut̪i waːst̪awaːniki komt̪a ɡamd̪araɡoːɭamɡaː @@ @@ weɖalpu @@ @@ et̪ːu"
tar2 = "35 mile akrut̪ɪ ʋast̪əʋanɪkɪ kont̪ə ɡənd̪ərəɡoːɭəŋɡa 36 mm ʋeɖəlpʊ 24 mm et̪t̪u"

In [247]:
src = "la fosse est soit chauffée avec des pierres chaudes provenant dun feu soit la chaleur géothermique réchauffe naturellement certaines zones du sol" 
tar = "la fɔsə  swat ʃofe avək de piɛrrə ʃodə prɔvənɑ̃t dœ̃ fœ swat la ʃalœr ʒeɔtɛrmikə reʃofə natyrələmɑ̃ sɛrtɛnə zɔn dy sɔl"

In [248]:
build_lexicon_entry(src1, tar1, latn=False)

{'35': {'spn'},
 'mile': {'m a j l'},
 'ఆకృతి': {'aː k r u t̪ i'},
 'వాస్తవానికి': {'w aː s t̪ a w aː n i k i'},
 'కొంత': {'k o m t̪ a'},
 'గందరగోళంగా': {'ɡ a m d̪ a r a ɡ oː ɭ a m ɡ aː'},
 '36': {'spn'},
 'mm': {'m'},
 'వెడల్పు': {'w e ɖ a l p u'},
 '24': {'spn'},
 'ఎత్తు': {'e t̪ː u'}}

In [254]:
import os

LATIN_LANG_PREFIXES = (
    "af_", "ast_", "az_", "bs_", "ca_", "ceb_", "cs_", "cy_", "da_", "de_", "en_", "es_", "et_", "ff_", "fi_", "fil_", "fr_", "ga_", "gl_", "ha_", "hr_", "hu_", "id_", "ig_", "is_", "it_", "jv_", "kam_", "kea_", "lb_", "lg_", "ln_", "lt_", "luo_", "lv_", "mi_", "ms_", "mt_", "nb_", "nl_", "nso_", "ny_", "oc_", "om_", "pl_", "pt_", "ro_", "sk_", "sl_", "sn_", "so_", "sv_", "sw_", "tr_", "umb_", "uz_", "vi_", "wo_", "xh_", "yo_", "zu_"
)


def build_mfa_dictionary(src_text_path: str, tar_text_path: str, output_name="lexicon.txt"):
    """
    Build an MFA-compatible pronunciation dictionary from parallel text files.
    """

    out_dir = os.path.dirname(src_text_path) or os.path.dirname(tar_text_path)
    output_path = os.path.join(out_dir, output_name)

    lexicon = {}
    toned = False
    for l in ('cmn_', 'vi_', 'yue_'):
        toned = toned or (l in src_text_path)
    latn = any(pre in src_text_path.replace('_input','') for pre in LATIN_LANG_PREFIXES)
    with open(src_text_path, 'r', encoding='utf-8') as f_src, \
         open(tar_text_path, 'r', encoding='utf-8') as f_tar:

        for line_num, (src_line, tar_line) in enumerate(zip(f_src, f_tar), 1):
            entry = build_lexicon_entry(src_line, tar_line, toned=toned, latn=latn)
            # merge sets (IMPORTANT CHANGE)
            for word, prons in entry.items():
                if word not in lexicon:
                    lexicon[word] = set()
                lexicon[word].update(prons)

    # Ensure <unk> exists
    if "<unk>" not in lexicon:
        lexicon["<unk>"] = {"spn"}

    # Write dictionary (tab-separated, multiple pronunciations)
    count = 0
    with open(output_path, 'w', encoding='utf-8') as f_out:
        for word in sorted(lexicon.keys()):
            for pron in lexicon[word]:
                if pron == 'spn':
                    count += 1
                f_out.write(f"{word}\t{pron}\n")
    print("spn count:",count, output_path)
    return output_path

In [255]:
build_mfa_dictionary('fleurs_ipa/th_th/train_sentences_input.txt', 'fleurs_ipa/th_th/train_sentences_epitran.txt', output_name="lexicon.txt")

spn count: 20 fleurs_ipa/th_th/lexicon.txt


'fleurs_ipa/th_th/lexicon.txt'

In [256]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()
meta_pd = pd.read_csv('lang_meta.tsv', sep='\t').fillna('')

def build_pron_dict(meta_row):
    global dyn
    dyn = {}
    lang = os.path.basename(meta_row['dir'])
    lang_pth = os.path.join('fleurs_ipa', lang)
    epi_code = meta_row['epitran']
    xpf = meta_row['xpf']
    src_pre = "train_sentences"
    src_pth = os.path.join(lang_pth, f"{src_pre}_input.txt")
    tar_pth = ""
    
    if xpf:
        tar_pth = os.path.join(lang_pth, f"{src_pre}_xpf.txt")
    elif epi_code:
        tar_pth = os.path.join(lang_pth, f"{src_pre}_epitran.txt")

    if os.path.isfile(tar_pth):
        build_mfa_dictionary(src_pth, tar_pth)

In [257]:
meta_pd.apply(build_pron_dict, axis=1)

spn count: 1 fleurs_ipa/af_za/lexicon.txt
spn count: 3 fleurs_ipa/am_et/lexicon.txt
spn count: 3 fleurs_ipa/ar_eg/lexicon.txt
spn count: 10 fleurs_ipa/ast_es/lexicon.txt
spn count: 1 fleurs_ipa/az_az/lexicon.txt
spn count: 5 fleurs_ipa/be_by/lexicon.txt
spn count: 5 fleurs_ipa/bg_bg/lexicon.txt
spn count: 9 fleurs_ipa/bn_in/lexicon.txt
spn count: 2 fleurs_ipa/ca_es/lexicon.txt
spn count: 1 fleurs_ipa/ceb_ph/lexicon.txt
spn count: 2 fleurs_ipa/ckb_iq/lexicon.txt
spn count: 7 fleurs_ipa/cmn_hans_cn/lexicon.txt
spn count: 1 fleurs_ipa/cs_cz/lexicon.txt
spn count: 1 fleurs_ipa/cy_gb/lexicon.txt
spn count: 1 fleurs_ipa/da_dk/lexicon.txt
spn count: 1 fleurs_ipa/de_de/lexicon.txt
spn count: 8 fleurs_ipa/el_gr/lexicon.txt
spn count: 1 fleurs_ipa/en_us/lexicon.txt
spn count: 2 fleurs_ipa/es_419/lexicon.txt
spn count: 2 fleurs_ipa/et_ee/lexicon.txt
spn count: 2 fleurs_ipa/fa_ir/lexicon.txt
spn count: 1 fleurs_ipa/ff_sn/lexicon.txt
spn count: 1 fleurs_ipa/fi_fi/lexicon.txt
spn count: 3 fleurs_ipa

0      None
1      None
2      None
3      None
4      None
       ... 
97     None
98     None
99     None
100    None
101    None
Length: 102, dtype: object

In [243]:
import pandas as pd

In [244]:
train_tsv = pd.read_csv("fleurs/data/af_za/train.tsv", sep='\t', names=['id','wav','txt', 'txt_norm', 'toks', 'spkr_id', 'spkr_gen'])

In [ ]:
len(train_tsv['spkr_id'].unique()), len(train_tsv)